# SIMT 编程实战：Gather 算子开发

## 概述

前面《编程模型》章节我们系统学习了 SIMT 的硬件架构、线程架构、核函数、内存层级、同步机制和编程 API。本节进入**实战**：用 Ascend C SIMT 从零实现一个一维 **Gather（采集）算子**，走完 Device 端核函数、Host 端调用、编译运行的完整流程。

Gather 是根据索引从张量中收集元素的一个算子，常用于特征提取、嵌入查找、动态切片等场景，广泛应用于图像、语言等多种领域的模型。Gather 是一个典型的**离散内存访问**算子——它根据一组索引，从输入向量里「东一个、西一个」地采集数据。这类访问模式不规整、分支灵活，正是 SIMT 编程相比 SIMD 的优势场景，我们使用最简单的一维 Gather 算子作为 SIMT 实战的第一个算子。

### 前置要求

- 已学习《编程模型》中 SIMT 的核函数、线程索引、内存层级等内容。
- 具备基础 C/C++ 和指针概念。
- 本教程的代码单元格用于写出源文件，实际编译执行需在 NPU 环境进行。

### 学习目标

学完本节后，你应该能够：

- 说明 Gather 算子的功能和数学表达式。
- 用 SIMT 思路设计 Gather 的数据切分（一个线程采集一个元素）。
- 写出 Gather 的 Device 端核函数，理解它如何复用线程索引和边界检查。
- 看懂 Host 端用 ACL Runtime API 完成内存分配、数据搬运、核函数启动和结果回传的完整流程。
- 用 CMake + 毕昇编译器编译并运行一个 SIMT 算子。

### 小节内容

- 环境准备
- Gather 算子功能介绍
- 算子设计：数据切分与接口
- Device 端核函数实现
- Host 端调用实现
- 编译与运行
- 小结

## 1. 环境准备

执行下方脚本，检查 CANN Toolkit 是否可用，并把 CANN 环境变量加载到当前 notebook 进程。

In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

print(f"CANN Toolkit: {cann_home}")


## 2. Gather 算子功能介绍

Gather（采集）算子根据索引张量 `index` 中的每个元素，从一维输入向量 `input` 中采集对应位置的数据，写入输出张量 `output`。数学表达式为：

```text
output[i] = input[index[i]]
```

举个例子，假设：

```text
input = [10, 11, 12, 13, 14]      // 输入数据
index = [0, 2, 2, 4, 1]           // 要采集的下标
output= [10, 12, 12, 14, 11]      // output[i] = input[index[i]]
```

`output[0] = input[index[0]] = input[0] = 10`，`output[1] = input[index[1]] = input[2] = 12`……以此类推。下图直观地展示了这个采集过程：

![Gather 算子功能示意](images/04_06_simt_gather_operator/gather_operator.svg)

它的特点是**离散访问**：`index` 里的下标是任意的，可能跳跃、可能重复（如上图中两个 `2` 都指向 `input[2]`），所以每个输出元素读的是输入里「不连续」的位置。这种不规整的访问，用 SIMT「一个线程独立处理一个元素」的方式表达起来非常自然——每个线程读自己那一个 `index[i]`，去取 `input[index[i]]`，互不相干。

**本节实现的算子规格**（与官方样例一致）：

| 项 | 取值 |
| --- | --- |
| 核函数名 | `gather_kernel` |
| 输入 `input` | 长度 100000，`float` |
| 输入 `index` | 长度 12288，`int32_t` |
| 输出 `output` | 长度 12288，`float` |

## 3. 算子设计：数据切分

### 一个线程采集一个元素

输出有 12288 个元素，我们就让 **12288 个线程**各采集一个元素——这正是 SIMT「一个线程算一份数据」的典型用法。按《线程架构》学过的层次组织：

- 每个线程块（Block）启用 **1024** 个线程（`threads_per_block = 1024`）。
- 线程块数使用向上取整公式计算：`blocks_per_grid = (index_total_length + threads_per_block - 1) / threads_per_block`，本例得到 **12** 个线程块。
- 总线程数 = 12 × 1024 = 12288，正好一个线程对应一个输出元素。

每个线程先用全局索引公式算出自己负责第几个元素：

```text
idx = blockIdx.x * blockDim.x + threadIdx.x
```

然后执行 `output[idx] = input[index[idx]]`。

> 本例中线程总数与输出元素数恰好相等，但通用做法是 `blocks_per_grid = (index_total_length + threads_per_block - 1) / threads_per_block` 向上取整，再用边界检查处理多余线程（见下方核函数里的 `if (idx >= index_total_length) return;`）。

另外，Gather 的数据搬入搬出**无需额外接口**，核函数内直接用指针 `[]` 访问 Global Memory 即可——这与《SIMT 内存层级》中「核函数通过指针直接访问全局内存」一致。

### 开始动手：建目录与引入头文件

接下来我们逐段把代码写入 `src/gather_1d/gather_1d.asc`（`.asc` 是 Ascend C 源文件后缀，Host 端和 Device 端代码写在同一个文件里）。第一个 `%%writefile` 覆盖创建文件，后续 `%%writefile -a` 追加。

先创建源码目录：

In [ ]:
!mkdir -p src/gather_1d

再写入文件头部需要的头文件（这一步**覆盖创建** `gather_1d.asc`）：

In [ ]:
%%writefile src/gather_1d/gather_1d.asc
// ===== 头文件引入 =====
#include <algorithm>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"   // ACL Runtime API

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);


## 4. Device 端核函数实现

文件创建好后，先写最核心的 Device 端核函数 `gather_kernel`。它的结构和《核函数》小节学的「算全局索引 → 边界检查 → 处理数据」三步完全一致：

- `__global__` 声明这是核函数，返回 `void`，三个指针参数都指向 Global Memory。
- **算全局索引**：`blockIdx.x * blockDim.x + threadIdx.x` 得到当前线程负责的输出元素编号 `idx`。
- **边界检查**：多余线程直接 `return`，防止越界（适配元素数不能被整除的通用场景）。
- **采集**：`index[idx]` 取出源下标，`input[index[idx]]` 据此取数写到 `output[idx]`，一行完成「间接寻址 + 采集」。

把核函数**追加**写入 `gather_1d.asc`：

In [ ]:
%%writefile -a src/gather_1d/gather_1d.asc

// ===== Device 端核函数 =====
// 每个线程采集一个元素：output[i] = input[index[i]]
__global__ void gather_kernel(float* input, int32_t* index, float* output, uint32_t index_total_length)
{
    // ① 算全局索引：当前线程负责第几个输出元素
    int32_t idx = blockIdx.x * blockDim.x + threadIdx.x;

    // ② 边界检查：多余线程直接退出，防止越界
    if (idx >= index_total_length) {
        return;
    }
    // ③ 采集：读 index[idx] 得到源下标，从 input 取出写回 output
    output[idx] = input[index[idx]];
}

## 5. Host 端调用实现

Host 端负责「准备数据 → 启动核函数 → 取回结果」，使用 ACL Runtime API，主要分为六步：

1. **初始化与创建 stream**：`aclInit` 初始化 ACL，`aclrtSetDevice` 选定 Device，`aclrtCreateStream` 创建任务流。
2. **分配内存**：Host 侧用 `aclrtMallocHost`，Device 侧用 `aclrtMalloc`，为 `input`/`index`/`output` 在 Device 上各分配一份。
3. **数据搬入**：用 `aclrtMemcpy`（`ACL_MEMCPY_HOST_TO_DEVICE`）把 `input`、`index` 拷到 Device。
4. **启动核函数**：用 `gather_kernel<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(...)` 下发计算，四个执行配置参数就是《核函数》里学过的那四个，本例每个线程块的线程数取 1024，线程块数由向上取整公式得到 12，动态共享内存大小取 0。
5. **同步等待**：核函数是**异步**下发到 stream 的，必须用 `aclrtSynchronizeStream` 等它执行完，否则可能在计算还没完成时就去读结果。
6. **数据搬出**：用 `aclrtMemcpy`（`ACL_MEMCPY_DEVICE_TO_HOST`）把结果拷回 Host，最后释放内存、`aclFinalize` 去初始化。

下面把这六步的完整实现**追加**写入 `gather_1d.asc`（用 `%%writefile -a`，代码里的注释标出了对应的步骤）：

In [ ]:
%%writefile -a src/gather_1d/gather_1d.asc

// ===== 第 2 段：Host 端调用函数 =====
std::vector<float> gather_1d(std::vector<float>& input, std::vector<int32_t>& index)
{
    if (input.empty() || index.empty()) {
        std::cout << "[ERROR] Empty input tensors." << std::endl;
        return {};
    }

    uint32_t input_total_length = input.size();
    uint32_t index_total_length = index.size();

    size_t input_total_byte_size = input_total_length * sizeof(float);
    size_t index_total_byte_size = index_total_length * sizeof(int32_t);
    size_t output_total_byte_size = index_total_length * sizeof(float);

    int32_t device_id = 0;
    aclrtStream stream = nullptr;

    uint8_t* input_host = reinterpret_cast<uint8_t *>(input.data());
    uint8_t* index_host = reinterpret_cast<uint8_t *>(index.data());
    uint8_t* output_host = nullptr;
    float* input_device = nullptr;
    int32_t* index_device = nullptr;
    float* output_device = nullptr;

    // 第 1 步：初始化与创建 stream
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(device_id));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 第 2 步：分配 Host / Device 内存；第 3 步：把输入从 Host 拷到 Device
    CHECK_ACL(aclrtMallocHost((void **)(&output_host), output_total_byte_size));
    CHECK_ACL(aclrtMalloc((void **)&input_device, input_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc((void **)&index_device, index_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc((void **)&output_device, output_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(input_device, input_total_byte_size, input_host, input_total_byte_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(index_device, index_total_byte_size, index_host, index_total_byte_size, ACL_MEMCPY_HOST_TO_DEVICE));

    // 第 4 步：配置执行参数并启动核函数
    uint32_t threads_per_block = 1024;  // 每个线程块的线程数（Block 大小）
    uint32_t blocks_per_grid = (index_total_length + threads_per_block - 1) / threads_per_block;  // 线程块数（Grid 大小）
    uint32_t dyn_ubuf_size = 0;        // 本样例无需动态共享内存
    gather_kernel<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(
        input_device, index_device, output_device, index_total_length);

    // 第 5 步：同步等待核函数执行完成
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 第 6 步：把结果从 Device 拷回 Host
    CHECK_ACL(aclrtMemcpy(output_host, output_total_byte_size, output_device, output_total_byte_size, ACL_MEMCPY_DEVICE_TO_HOST));
    std::vector<float> output((float *)output_host, (float *)(output_host + output_total_byte_size));

    // 释放内存
    CHECK_ACL(aclrtFree(input_device));
    CHECK_ACL(aclrtFree(index_device));
    CHECK_ACL(aclrtFree(output_device));
    CHECK_ACL(aclrtFreeHost(output_host));

    const char* err = aclGetRecentErrMsg();
    if (err != nullptr) {
        fprintf(stderr, "%s\n", err);
    }

    // 去初始化
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(device_id));
    CHECK_ACL(aclFinalize());
    return output;
}

## 6. 测试数据与验证

最后补上 `main` 和结果校验，让算子能独立跑起来自测：

- `main` 构造长度 100000 的 `input`（`input[i] = i * 1.2`）和长度 12288 的 `index`（取模保证落在 `input` 范围内），并在 CPU 上算出预期结果 `golden`。
- 调用 `gather_1d` 得到 NPU 的计算结果 `output`，再用 `verify_result` 和 `golden` 逐元素对比。

把这部分**追加**写入 `gather_1d.asc`，至此整个 `.asc` 文件就组装完成了：

In [ ]:
%%writefile -a src/gather_1d/gather_1d.asc

// ===== 第 3 段：结果校验与 main =====
uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_tensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    print_tensor(output, "Output");
    print_tensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t input_total_length = 100000;
    constexpr uint32_t index_total_length = 12288;

    // 构造输入数据
    std::vector<float> input(input_total_length);
    for (uint32_t i = 0; i < input_total_length; i++) {
        input[i] = i * 1.2f;
    }

    // 构造索引（取模保证落在 input 范围内）
    std::vector<int32_t> index(index_total_length);
    for (uint32_t i = 0; i < index_total_length; i++) {
        index[i] = static_cast<int32_t>((i * 97 + 13) % input_total_length);
    }

    // CPU 算出预期结果 golden
    std::vector<float> golden(index_total_length);
    for (uint32_t i = 0; i < index_total_length; i++) {
        golden[i] = input[index[i]];
    }

    // 调用 NPU 算子并校验
    std::vector<float> output = gather_1d(input, index);
    if (output.empty()) {
        return 1;
    }
    return verify_result(output, golden);
}

## 7. 编译与运行

### CMake 配置

用 CMake + 毕昇编译工具链编译。关键是声明 `ASC` 语言、设置 NPU 架构，并加上 `--enable-simt` 启用 SIMT 编程场景。运行下方单元格生成 `CMakeLists.txt`：

In [ ]:
%%writefile src/gather_1d/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

# find_package(ASC) 用于查找和配置 Ascend C 编译工具链
find_package(ASC REQUIRED)

# 声明项目支持 ASC（毕昇编译器编译 Ascend C）和 CXX 两种语言
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    gather_1d.asc
)

# 通过编译选项设置 NPU 架构
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)

### 编译并运行

在 `src/gather_1d` 目录下执行编译运行三连：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd src/gather_1d && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，`main` 会构造测试数据、调用算子、并和 CPU 算出的 `golden` 对比，预期输出：

```text
Output: 15.6 132 ...
Golden: 15.6 132 ...
[Success] Case accuracy is verification passed.
```

看到 `[Success] Case accuracy is verification passed.` 就说明算子计算结果与预期完全一致。

**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定 NPU 架构版本，`dav-` 后为架构号，需替换为实际型号对应的版本 |
| `--enable-simt` | **启用 SIMT 编程场景**，编译 SIMT 算子必须加 |

### 使用 msprof op 采集性能

完成正确性验证后，可以继续使用 `msprof op` 对当前 `threads_per_block = 1024` 的版本采集算子性能。`msprof op ./demo` 会运行可执行文件并生成 `OPPROF_{timestamp}_...` 性能数据目录，常用来查看算子基础信息、执行耗时、Pipe 利用率和内存访问情况。


In [ ]:
# 采集 threads_per_block = 1024 的算子性能
!cd src/gather_1d/build && msprof op ./demo


采集完成后，记录本次 `OPPROF_{timestamp}_...` 目录以及关键指标。随后将 `src/gather_1d/gather_1d.asc` 中的 `threads_per_block` 从 `1024` 改为 `256`，重新编译并再次执行 `msprof op ./demo`。

本教程在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 配置 | 每个线程块的线程数 | 线程块数（Block Dim） | Task Duration(us) |
| --- | --- | --- | --- |
| 版本 A | 1024 | 12 | 3.926 |
| 版本 B | 256 | 48 | 3.119 |

从本次采集看，将每个线程块的线程数从 1024 调整为 256 后，线程块数从 12 增加到 48，`Task Duration` 从 3.926 us 降到 3.119 us，耗时约下降 20.56%。这说明在该 Gather 配置下，更多线程块带来了更好的并行调度效果。

两次 `msprof op` 的 Performance Summary 均提示 `MTE2 bandwidth utilization lower than 80% when active`，说明还可以继续从访存行为、数据布局或线程粒度等方向分析优化空间。进一步分析时，可以打开 `OpBasicInfo.csv` 查看基础耗时和 `PipeUtilization.csv` 查看 Vector、Scalar、MTE2/MTE3 等流水占比。


In [ ]:
# 将 gather_1d.asc 中 threads_per_block 改为 256 后，重新编译并采集性能
!cd src/gather_1d && mkdir -p build_tpb256 && cd build_tpb256 && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 msprof op ./demo


## 小结

本节用一维 Gather 算子完整走了一遍 SIMT 算子开发流程，把前面《编程模型》学的概念落到了实处：

- **算子功能**：`output[i] = input[index[i]]`，根据索引从输入里离散采集数据。
- **数据切分**：一个线程采集一个元素，每个线程块的线程数为 1024，按向上取整公式启动 12 个线程块，正好覆盖 12288 个输出元素。
- **Device 端核函数**：沿用「算全局索引（`blockIdx.x * blockDim.x + threadIdx.x`）→ 边界检查 → 处理数据」三步结构，核心只有一行 `output[idx] = input[index[idx]]`，直接用指针访问 Global Memory。
- **Host 端调用**：用 ACL Runtime API 完成「初始化 → 分配内存 → 数据搬入 → `<<<>>>` 启动核函数 → 同步 → 数据搬出」。
- **编译运行**：用 CMake + 毕昇编译器，关键编译选项是 `--npu-arch` 指定架构、`--enable-simt` 启用 SIMT。

Gather 是离散访问类算子的代表，它展示了 SIMT「一个线程一份数据、各自独立寻址」的天然优势。掌握这个最小可运行样例后，你就可以在此基础上扩展到更复杂的算子——比如多维 Gather、带计算的 Gather，或尝试用前面学过的 Warp 函数、共享内存做进一步优化。

## 课后编程习题：二维 Gather 算子

本节已经实现了一维 Gather。请参考《SIMT 线程架构》《SIMT 核函数》《SIMT 内存层级》的内容，自行实现一个**二维 Gather 算子**。

本练习的二维 Gather 语义是：从二维输入矩阵 `input` 中，根据一维 `index` 采集指定的整行数据。`index[i]` 表示要从 `input` 中采集哪一行，输出 `output` 的第 `i` 行等于 `input[index[i]]`。

```text
input  shape: [100000, 128]
index  shape: [12288]
output shape: [12288, 128]

output[i, col] = input[index[i], col]
```

建议使用如下固定规格完成练习：

| 项 | 取值 |
| --- | --- |
| 核函数名 | `gather_2d_custom` |
| 输入 `input` | `[100000, 128]`，`float` |
| 输入 `index` | `[12288]`，`uint32_t` |
| 输出 `output` | `[12288, 128]`，`float` |
| `in_width` | 128 |
| `index_total_length` | 12288 |
| 线程块数 | 48 |
| 每个线程块的线程数 | 256 |

实现时请完成以下要点：

1. 一个线程处理输出的一整行，即处理 `output[out_row, 0:128]`。
2. 用一维全局线程索引计算输出行号：`out_row = blockIdx.x * blockDim.x + threadIdx.x`。
3. 对 `out_row >= index_total_length` 的多余线程做边界检查。
4. 读取 `in_row = index[out_row]`，然后循环 `col = 0..in_width-1`。
5. 用二维展平地址完成整行采集：`input_idx = in_row * in_width + col`，`output_idx = out_row * in_width + col`。


In [ ]:
!mkdir -p src/gather_2d

Host 侧代码已经写好，下面的单元格会生成完整的 `gather_2d.asc` 骨架；请只补充 Device 端 `gather_2d_custom` 函数体中的 TODO。

In [ ]:
%%writefile src/gather_2d/gather_2d.asc
// ===== 头文件引入 =====
#include <algorithm>
#include <cstdint>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

// ===== Device 端核函数 =====
// input:  [100000, 128]
// index:  [12288]
// output: [12288, 128]
// TODO: 只需要补充这个核函数的函数体。
__global__ void gather_2d_custom(float* input, uint32_t* index, float* gather_output,
    uint32_t in_width, uint32_t index_total_length)
{
    // TODO: 1. 计算输出行号 out_row。
    // TODO: 2. 对 out_row 做边界检查。
    // TODO: 3. 读取输入行号 in_row = index[out_row]。
    // TODO: 4. 循环 col = 0..in_width-1，完成整行采集。
    // 删除下一行 return 后完成实现。
    return;
}

// ===== Host 端调用函数：已写好，无需修改 =====
std::vector<float> gather_2d(std::vector<float>& input, const uint32_t* in_shape, std::vector<uint32_t>& index)
{
    if (input.size() != in_shape[0] * in_shape[1] || index.empty()) {
        std::cout << "[ERROR] Invalid input tensors." << std::endl;
        return {};
    }

    uint32_t input_total_length = input.size();
    size_t input_total_byte_size = input_total_length * sizeof(float);

    uint32_t index_total_length = index.size();
    size_t index_total_byte_size = index_total_length * sizeof(uint32_t);

    uint32_t output_total_length = index_total_length * in_shape[1];
    size_t output_total_byte_size = output_total_length * sizeof(float);

    int32_t device_id = 0;
    aclrtStream stream = nullptr;

    uint8_t* input_host = reinterpret_cast<uint8_t *>(input.data());
    uint8_t* index_host = reinterpret_cast<uint8_t *>(index.data());
    uint8_t* output_host = nullptr;
    float* input_device = nullptr;
    uint32_t* index_device = nullptr;
    float* output_device = nullptr;

    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(device_id));
    CHECK_ACL(aclrtCreateStream(&stream));

    CHECK_ACL(aclrtMallocHost(reinterpret_cast<void **>(&output_host), output_total_byte_size));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void **>(&input_device), input_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void **>(&index_device), index_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void **>(&output_device), output_total_byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(input_device, input_total_byte_size, input_host, input_total_byte_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(index_device, index_total_byte_size, index_host, index_total_byte_size, ACL_MEMCPY_HOST_TO_DEVICE));

    uint32_t blocks_per_grid = 48;
    uint32_t threads_per_block = 256;
    uint32_t dyn_ubuf_size = 0;
    gather_2d_custom<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(
        input_device, index_device, output_device, in_shape[1], index_total_length);

    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtMemcpy(output_host, output_total_byte_size, output_device, output_total_byte_size, ACL_MEMCPY_DEVICE_TO_HOST));
    std::vector<float> output(reinterpret_cast<float *>(output_host),
        reinterpret_cast<float *>(output_host + output_total_byte_size));

    CHECK_ACL(aclrtFree(input_device));
    CHECK_ACL(aclrtFree(index_device));
    CHECK_ACL(aclrtFree(output_device));
    CHECK_ACL(aclrtFreeHost(output_host));

    const char* err = aclGetRecentErrMsg();
    if (err != nullptr) {
        fprintf(stderr, "%s\n", err);
    }

    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(device_id));
    CHECK_ACL(aclFinalize());
    return output;
}

uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_tensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    print_tensor(output, "Output");
    print_tensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    }
    std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
    return 1;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t in_shape[2] = {100000, 128};
    constexpr uint32_t input_total_length = in_shape[0] * in_shape[1];
    std::vector<float> input(input_total_length);
    for (uint32_t i = 0; i < input_total_length; i++) {
        input[i] = i * 1.2f;
    }

    constexpr uint32_t index_total_length = 12288;
    std::vector<uint32_t> index(index_total_length);
    for (uint32_t i = 0; i < index_total_length; i++) {
        if (i % 2 == 0) {
            index[i] = i + 5;
        } else {
            index[i] = i + 3;
        }
    }

    std::vector<float> golden(index_total_length * in_shape[1]);
    for (uint32_t i = 0; i < index_total_length; i++) {
        for (uint32_t j = 0; j < in_shape[1]; j++) {
            golden[i * in_shape[1] + j] = input[index[i] * in_shape[1] + j];
        }
    }

    std::vector<float> output = gather_2d(input, in_shape, index);
    if (output.empty()) {
        return 1;
    }
    return verify_result(output, golden);
}


再创建 CMake 配置`：

In [ ]:
%%writefile src/gather_2d/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    gather_2d.asc
)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)

target_link_libraries(demo PRIVATE
    m
    platform
    tiling_api
    dl
)


完成 `gather_2d.asc` 后，在 NPU 环境中执行以下命令验证结果：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd src/gather_2d && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

如果实现正确，输出应包含：

```text
[Success] Case accuracy is verification passed.
```

可以执行下面的单元格查看参考实现。建议先独立完成，再对照参考答案检查线程索引、边界检查和二维展平地址计算。

In [ ]:
!cat answer/04_06_simt_gather_operator/gather_2d.asc

In [ ]:
!cat answer/04_06_simt_gather_operator/CMakeLists.txt

完成本练习后，可以对比一维 Gather 和二维 Gather：一维版本中每个线程采集一个标量，二维版本中每个线程先根据 `index[out_row]` 找到输入行，再用一个列循环采集整行数据。这个变化体现了 SIMT 中“线程负责粒度”可以按算子语义调整：既可以一个线程处理一个元素，也可以一个线程处理一小段连续数据。